In [1]:
import numpy as np
import pandas as pd

In [2]:
from boostie.model   import boostieModel
from boostie.data    import (
    train_test_split,
)
from boostie.metrics import (
    rmse, mae, r_squared,
    log_loss, accuracy,
    confusion_matrix, precision_recall,
)

## Read Titanic CSV

In [3]:
df_raw = pd.read_csv('./resources/titanic.csv')

In [4]:
df_raw.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df_raw.Embarked.value_counts()

Embarked
S    644
C    168
Q     77
Name: count, dtype: int64

In [27]:
df = df_raw.loc[df_raw["Age"] >= 0, :].reset_index(drop=True)

## Define model

In [28]:
model = boostieModel(
        n_estimators  = 100,
        max_depth     = 3,
        learning_rate = 0.1,
        reg_lambda    = 1.0,
        reg_gamma     = 0.0,
        objective     = "tweedie",
    )

## Preprocess cols

In [29]:
preproccess={"Embarked": "one_hot_encoding", "Sex": "one_hot_encoding"}

In [30]:
preprocessed_df = model.preprocess(df, feature=preproccess, dropna=False, inplace=True)

In [31]:
preprocessed_df.head()

,PassengerId,Survived,Pclass,Name,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked_S,Embarked_C,Embarked_Q,Embarked__missing,Sex_male,Sex_female
0,1,0,3,"Braund, Mr. Owen Harris",22.0,1,0,A/5 21171,7.25,NaN,1.0,0.0,0.0,0.0,1.0,0.0
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",38.0,1,0,PC 17599,71.2833,C85,0.0,1.0,0.0,0.0,0.0,1.0
2,3,1,3,"Heikkinen, Miss. Laina",26.0,0,0,STON/O2. 3101282,7.925,NaN,1.0,0.0,0.0,0.0,0.0,1.0
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",35.0,1,0,113803,53.1,C123,1.0,0.0,0.0,0.0,0.0,1.0
4,5,0,3,"Allen, Mr. William Henry",35.0,0,0,373450,8.05,NaN,1.0,0.0,0.0,0.0,1.0,0.0


In [32]:
preprocessed_df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Age', 'SibSp', 'Parch',
       'Ticket', 'Fare', 'Cabin', 'Embarked_S', 'Embarked_C', 'Embarked_Q',
       'Embarked__missing', 'Sex_male', 'Sex_female'],
      dtype='str')

## Train Test Split

In [33]:
TARGET = "Age"

features = ["SibSp", "Parch", 'Embarked_S',
       'Embarked_C', 'Embarked_Q', 'Embarked__missing', 'Sex_male',
       'Sex_female']

In [34]:
X_train, X_test, y_train, y_test = train_test_split(preprocessed_df[features], preprocessed_df[TARGET], test_size=0.3, seed=69)

In [35]:
X_train

,SibSp,Parch,Embarked_S,Embarked_C,Embarked_Q,Embarked__missing,Sex_male,Sex_female
0,0,0,1.0,0.0,0.0,0.0,0.0,1.0
1,0,0,1.0,0.0,0.0,0.0,0.0,1.0
2,0,2,1.0,0.0,0.0,0.0,1.0,0.0
3,0,0,1.0,0.0,0.0,0.0,1.0,0.0
4,1,0,1.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...
494,1,1,0.0,1.0,0.0,0.0,1.0,0.0
495,0,0,1.0,0.0,0.0,0.0,1.0,0.0
496,0,0,1.0,0.0,0.0,0.0,1.0,0.0
497,0,0,1.0,0.0,0.0,0.0,1.0,0.0


In [36]:
y_train

0      45.0
1      58.0
2      36.5
3      28.0
4      28.0
       ... 
494    60.0
495    21.0
496    17.0
497    22.0
498    36.0
Length: 499, dtype: object

## Train

In [37]:
model.fit(X_train, y_train, verbose=True)

  [round   10/100]  train loss: 145.353228
  [round   20/100]  train loss: 26.309983
  [round   30/100]  train loss: 9.572223
  [round   40/100]  train loss: 6.833476
  [round   50/100]  train loss: 6.227669
  [round   60/100]  train loss: 6.059207
  [round   70/100]  train loss: 5.999637
  [round   80/100]  train loss: 5.972150
  [round   90/100]  train loss: 5.946261
  [round  100/100]  train loss: 5.929784


XGBoostModel(objective='tweedie', n_estimators=100, max_depth=3, lr=0.1, lambda=1.0, gamma=0.0)

## Evaluate

In [38]:
probs = model.predict(X_test)
print(f"  MAE  : {mae(y_test, probs):.4f}")
print(f"  RMSE  : {rmse(y_test, probs):.4f}")

  MAE  : 10.8816
  RMSE  : 13.1625


In [42]:
importances = model.feature_importances(len(model.feature_names))
for i, imp in enumerate(importances):
    bar = "█" * int(imp * 40)
    print(f"  {model.feature_names[i]}  {imp:.3f}  {bar}")

  SibSp  0.317  ████████████
  Parch  0.343  █████████████
  Embarked_S  0.037  █
  Embarked_C  0.045  █
  Embarked_Q  0.045  █
  Embarked__missing  0.057  ██
  Sex_male  0.101  ████
  Sex_female  0.056  ██
